<a href="https://colab.research.google.com/github/sunainakhatwani12/flyrank-ml-internship-sunaina/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunainakhatwani12/flyrank-ml-internship-sunaina/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. Method choice and why

My confirmed lane is **Refresh / Content Opportunity Scoring**.

The decision question is:

> Which content pages should a content or SEO team review first for a possible refresh?

This is a ranking problem. I need a score for every page so that pages with stronger measured evidence of decline can appear near the top of a review queue.

I will start with **Logistic Regression** because it provides a simple and readable benchmark. I will then train a **Random Forest classifier** because it can learn non-linear relationships and interactions between signals such as content age, staleness, impressions, clicks, search position, engagement, and content type.

I will use each model's predicted probability of the declining label as its ranking score. A higher probability places the page higher in the review queue.

The main metrics are:

- Precision@20
- Precision@50
- Precision@100

These metrics fit the decision because a content team usually reviews only the first part of a ranked queue. I will also report average precision and ROC-AUC as supporting classification metrics.

The target is `is_declining_label`, created from `trend_direction` only for evaluation and model training.

To prevent direct label leakage, the models do not use:

- `trend_direction`
- `trend_pct`
- `is_declining_label` as an input
- Last-30-day versus previous-30-day outcome columns
- Existing FlyRank action flags
- `content_id` or `client_id` as predictive features

The model output is decision support. It does not prove that refreshing a page will improve performance, and it should not automatically trigger a content change.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# ML-08 — SECTION 2
# Load data and create an honest grouped split
# ============================================================

from pathlib import Path
import os
import json
import shutil
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.base import clone

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_SEED = 42

# ------------------------------------------------------------
# 1. Find the repository and starter CSV
# ------------------------------------------------------------

REPO_NAME = "flyrank-ml-internship-sunaina"
REPO_URL = (
    "https://github.com/"
    "sunainakhatwani12/"
    "flyrank-ml-internship-sunaina.git"
)

possible_roots = [
    Path.cwd(),
    Path("/content") / REPO_NAME,
    Path("/content"),
]

repo_root = None

for root in possible_roots:
    candidate = root / "data" / "raw" / "content_refresh_anonymized.csv"

    if candidate.exists():
        repo_root = root
        break

if repo_root is None:
    clone_path = Path("/content") / REPO_NAME

    if clone_path.exists():
        shutil.rmtree(clone_path)

    print("Repository files were not found in this Colab session.")
    print("Cloning the public repository...")

    subprocess.run(
        ["git", "clone", REPO_URL, str(clone_path)],
        check=True,
    )

    repo_root = clone_path

os.chdir(repo_root)

data_path = (
    repo_root
    / "data"
    / "raw"
    / "content_refresh_anonymized.csv"
)

if not data_path.exists():
    raise FileNotFoundError(
        f"Starter CSV was not found at:\n{data_path}\n\n"
        "Confirm that data/raw/content_refresh_anonymized.csv "
        "exists in the repository."
    )

print(f"Repository root: {repo_root}")
print(f"Data path: {data_path}")

# ------------------------------------------------------------
# 2. Load and verify the starter data
# ------------------------------------------------------------

df = pd.read_csv(data_path)

required_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "days_since_last_update",
    "impressions_90d",
]

missing_required = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_required:
    raise ValueError(
        f"Required columns are missing: {missing_required}"
    )

# Build the observed decline label.
# trend_direction and trend_pct will never be model features.
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# avg_position = 0 means no position data, not rank zero.
if "avg_position" in df.columns:
    df["avg_position_no_data"] = (
        df["avg_position"].fillna(0).eq(0)
    ).astype(int)

    df.loc[
        df["avg_position"].eq(0),
        "avg_position",
    ] = np.nan

print("\nDATA CHECK")
print("-" * 70)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Unique content IDs: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique():,}")
print(
    "Observed declining pages: "
    f"{df['is_declining_label'].sum():,}"
)
print(
    "Overall decline base rate: "
    f"{df['is_declining_label'].mean():.2%}"
)

# ------------------------------------------------------------
# 3. Select allowed model features
# ------------------------------------------------------------

candidate_numeric_features = [
    "days_since_last_update",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "ctr",
    "avg_position",
    "avg_position_no_data",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "word_count",
    "char_count",
    "search_volume",
    "competition",
    "cpc",
]

candidate_categorical_features = [
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

numeric_features = [
    column
    for column in candidate_numeric_features
    if column in df.columns
]

categorical_features = [
    column
    for column in candidate_categorical_features
    if column in df.columns
]

feature_columns = (
    numeric_features
    + categorical_features
)

forbidden_features = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "needs_refresh",
    "needs_ctr_fix",
    "is_quick_win",
    "refresh_flag",
    "action_flag",
}

leaked_features = sorted(
    set(feature_columns).intersection(forbidden_features)
)

assert leaked_features == [], (
    f"Forbidden model inputs found: {leaked_features}"
)

assert "trend_direction" not in feature_columns
assert "trend_pct" not in feature_columns
assert "is_declining_label" not in feature_columns
assert "content_id" not in feature_columns
assert "client_id" not in feature_columns

print("\nFEATURE CHECK")
print("-" * 70)
print(f"Numeric features ({len(numeric_features)}):")
print(numeric_features)

print(
    f"\nCategorical features "
    f"({len(categorical_features)}):"
)
print(categorical_features)

print("\nForbidden predictive inputs found: NONE")

# ------------------------------------------------------------
# 4. Grouped client split
# ------------------------------------------------------------

X = df[feature_columns].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_SEED,
)

train_index, test_index = next(
    group_split.split(
        X,
        y,
        groups=groups,
    )
)

X_train = X.iloc[train_index].copy()
X_test = X.iloc[test_index].copy()

y_train = y.iloc[train_index].copy()
y_test = y.iloc[test_index].copy()

train_meta = df.iloc[train_index][
    ["content_id", "client_id"]
].copy()

test_meta = df.iloc[test_index][
    ["content_id", "client_id"]
].copy()

train_clients = set(train_meta["client_id"])
test_clients = set(test_meta["client_id"])

client_overlap = train_clients.intersection(test_clients)

assert len(client_overlap) == 0, (
    "Client leakage detected between train and test."
)

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

print("\nGROUPED SPLIT CHECK")
print("-" * 70)
print(f"Random seed: {RANDOM_SEED}")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Training clients: {len(train_clients):,}")
print(f"Test clients: {len(test_clients):,}")
print(f"Client overlap: {len(client_overlap)}")
print(
    "Training decline base rate: "
    f"{y_train.mean():.2%}"
)
print(
    "Test decline base rate: "
    f"{y_test.mean():.2%}"
)

split_summary = pd.DataFrame(
    {
        "split": ["Training", "Test"],
        "rows": [len(X_train), len(X_test)],
        "clients": [
            len(train_clients),
            len(test_clients),
        ],
        "declining_pages": [
            int(y_train.sum()),
            int(y_test.sum()),
        ],
        "decline_base_rate": [
            float(y_train.mean()),
            float(y_test.mean()),
        ],
    }
)

display(split_summary)

Repository root: /content/flyrank-ml-internship-sunaina
Data path: /content/flyrank-ml-internship-sunaina/data/raw/content_refresh_anonymized.csv

DATA CHECK
----------------------------------------------------------------------
Rows: 30,000
Columns: 46
Unique content IDs: 30,000
Unique clients: 32
Observed declining pages: 16,262
Overall decline base rate: 54.21%

FEATURE CHECK
----------------------------------------------------------------------
Numeric features (23):
['days_since_last_update', 'content_age_days', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'ctr', 'avg_position', 'avg_position_no_data', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'word_count', 'char_count', 'search_volume', 'competition', 'cpc']

Categorical features (8):
['content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_t

,split,rows,clients,declining_pages,decline_base_rate
0,Training,22885,24,12587,0.5500
1,Test,7115,8,3675,0.5165


## 2. Split design

I use a grouped train/test split based on `client_id`.

Approximately 75% of the clients are placed in the training set and 25% are placed in the test set. All rows belonging to one client stay in only one side of the split.

This is more honest than a random row split because pages from the same client may share content strategy, audience, publishing patterns, and measurement behaviour. A random row split could allow the model to learn client-specific patterns from one page and then appear successful on another page from the same client.

The test clients are therefore unseen during model training.

My Week 4 notebook measured the original baseline on the complete dataset. For this Week 5 comparison, I recompute the unchanged Week 4 rule on the held-out test set. The baseline and both models are therefore compared on the same test rows, using the same target and the same Precision@K metrics.

The random seed is fixed at 42 so that rerunning the notebook reproduces the same split and results.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# ML-08 — SECTION 3
# Train models and compare against the Week 4 baseline
# ============================================================

# ------------------------------------------------------------
# 1. Week 4 baseline scoring function
# ------------------------------------------------------------

STALE_THRESHOLD_DAYS = 180
VISIBILITY_THRESHOLD_IMPRESSIONS = 300

# Caps are learned from training rows only.
staleness_cap = max(
    STALE_THRESHOLD_DAYS + 1,
    float(
        X_train["days_since_last_update"]
        .quantile(0.95)
    ),
)

impression_cap = max(
    VISIBILITY_THRESHOLD_IMPRESSIONS + 1,
    float(
        X_train["impressions_90d"]
        .quantile(0.95)
    ),
)


def calculate_baseline_score(data):
    """
    Reproduce the Week 4 rule without using the label.
    """

    result = pd.DataFrame(
        index=data.index
    )

    result["is_stale"] = (
        data["days_since_last_update"]
        >= STALE_THRESHOLD_DAYS
    ).astype(int)

    result["has_meaningful_visibility"] = (
        data["impressions_90d"]
        >= VISIBILITY_THRESHOLD_IMPRESSIONS
    ).astype(int)

    result["eligible_for_refresh"] = (
        (result["is_stale"] == 1)
        & (
            result[
                "has_meaningful_visibility"
            ] == 1
        )
    ).astype(int)

    staleness_denominator = (
        staleness_cap
        - STALE_THRESHOLD_DAYS
    )

    result["staleness_component"] = (
        data["days_since_last_update"]
        .clip(
            lower=STALE_THRESHOLD_DAYS,
            upper=staleness_cap,
        )
        .sub(STALE_THRESHOLD_DAYS)
        .div(staleness_denominator)
        .mul(60)
    )

    log_floor = np.log1p(
        VISIBILITY_THRESHOLD_IMPRESSIONS
    )
    log_cap = np.log1p(impression_cap)

    result["visibility_component"] = (
        np.log1p(
            data["impressions_90d"].clip(
                lower=(
                    VISIBILITY_THRESHOLD_IMPRESSIONS
                ),
                upper=impression_cap,
            )
        )
        .sub(log_floor)
        .div(log_cap - log_floor)
        .mul(40)
    )

    ineligible = (
        result["eligible_for_refresh"] == 0
    )

    result.loc[
        ineligible,
        "staleness_component",
    ] = 0

    result.loc[
        ineligible,
        "visibility_component",
    ] = 0

    result["baseline_action_score"] = (
        result["staleness_component"]
        + result["visibility_component"]
    ).clip(0, 100)

    return result


baseline_test = calculate_baseline_score(
    X_test
)

baseline_test_scores = baseline_test[
    "baseline_action_score"
].copy()

assert baseline_test_scores.between(
    0,
    100,
).all()

print("WEEK 4 BASELINE ON THE HELD-OUT TEST SET")
print("-" * 70)
print(
    "Eligible test candidates: "
    f"{baseline_test['eligible_for_refresh'].sum():,}"
)
print(
    "Maximum baseline score: "
    f"{baseline_test_scores.max():.2f}"
)

# ------------------------------------------------------------
# 2. Preprocessing
# ------------------------------------------------------------

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
        (
            "scaler",
            StandardScaler(
                with_mean=False
            ),
        ),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features,
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features,
        ),
    ],
    remainder="drop",
)

# ------------------------------------------------------------
# 3. Logistic Regression
# ------------------------------------------------------------

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor),
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)

print("\nTraining Logistic Regression...")

logistic_model.fit(
    X_train,
    y_train,
)

logistic_test_scores = (
    logistic_model.predict_proba(
        X_test
    )[:, 1]
)

# ------------------------------------------------------------
# 4. Random Forest
# ------------------------------------------------------------

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor),
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=10,
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=-1,
            ),
        ),
    ]
)

print("Training Random Forest...")

random_forest_model.fit(
    X_train,
    y_train,
)

random_forest_test_scores = (
    random_forest_model.predict_proba(
        X_test
    )[:, 1]
)

print("Model training complete.")

# ------------------------------------------------------------
# 5. Ranking metric functions
# ------------------------------------------------------------

def make_ranked_results(
    scores,
    y_true,
    metadata,
    extra_data=None,
    score_name="ranking_score",
    baseline_tie_break=False,
):
    """
    Combine scores with labels and rank highest first.
    """

    ranked = metadata.copy()

    ranked["actual_label"] = (
        y_true.to_numpy()
    )

    ranked[score_name] = np.asarray(
        scores
    )

    if extra_data is not None:
        for column in extra_data.columns:
            ranked[column] = (
                extra_data[column]
                .to_numpy()
            )

    sort_columns = [score_name]
    ascending = [False]

    if baseline_tie_break:
        sort_columns.extend(
            [
                "impressions_90d",
                "days_since_last_update",
            ]
        )
        ascending.extend(
            [False, False]
        )

    ranked = (
        ranked.sort_values(
            by=sort_columns,
            ascending=ascending,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranked.insert(
        0,
        "rank",
        np.arange(
            1,
            len(ranked) + 1,
        ),
    )

    return ranked


def precision_at_k(
    ranked_data,
    k,
    label_column="actual_label",
):
    actual_k = min(
        k,
        len(ranked_data),
    )

    if actual_k == 0:
        return np.nan

    return float(
        ranked_data
        .head(actual_k)[label_column]
        .mean()
    )


baseline_extra = X_test[
    [
        "impressions_90d",
        "days_since_last_update",
    ]
].copy()

baseline_ranked = make_ranked_results(
    scores=baseline_test_scores,
    y_true=y_test,
    metadata=test_meta,
    extra_data=baseline_extra,
    score_name="baseline_action_score",
    baseline_tie_break=True,
)

logistic_ranked = make_ranked_results(
    scores=logistic_test_scores,
    y_true=y_test,
    metadata=test_meta,
    score_name="model_probability",
)

random_forest_ranked = make_ranked_results(
    scores=random_forest_test_scores,
    y_true=y_test,
    metadata=test_meta,
    score_name="model_probability",
)

# ------------------------------------------------------------
# 6. Comparison table
# ------------------------------------------------------------

test_base_rate = float(
    y_test.mean()
)


def safe_roc_auc(y_true, scores):
    if y_true.nunique() < 2:
        return np.nan

    return float(
        roc_auc_score(
            y_true,
            scores,
        )
    )


def build_metric_row(
    method_name,
    ranked_data,
    scores,
):
    return {
        "method": method_name,
        "test_base_rate": test_base_rate,
        "precision_at_20": precision_at_k(
            ranked_data,
            20,
        ),
        "precision_at_50": precision_at_k(
            ranked_data,
            50,
        ),
        "precision_at_100": precision_at_k(
            ranked_data,
            100,
        ),
        "average_precision": float(
            average_precision_score(
                y_test,
                scores,
            )
        ),
        "roc_auc": safe_roc_auc(
            y_test,
            scores,
        ),
    }


comparison_table = pd.DataFrame(
    [
        build_metric_row(
            "Week 4 rule baseline",
            baseline_ranked,
            baseline_test_scores,
        ),
        build_metric_row(
            "Logistic Regression",
            logistic_ranked,
            logistic_test_scores,
        ),
        build_metric_row(
            "Random Forest",
            random_forest_ranked,
            random_forest_test_scores,
        ),
    ]
)

comparison_table[
    "lift_at_20_vs_base_rate"
] = (
    comparison_table["precision_at_20"]
    - comparison_table["test_base_rate"]
)

comparison_table[
    "lift_at_50_vs_base_rate"
] = (
    comparison_table["precision_at_50"]
    - comparison_table["test_base_rate"]
)

comparison_table[
    "lift_at_100_vs_base_rate"
] = (
    comparison_table["precision_at_100"]
    - comparison_table["test_base_rate"]
)

print("\nHONEST COMPARISON TABLE")
print("-" * 70)

display(
    comparison_table.style.format(
        {
            "test_base_rate": "{:.2%}",
            "precision_at_20": "{:.2%}",
            "precision_at_50": "{:.2%}",
            "precision_at_100": "{:.2%}",
            "average_precision": "{:.4f}",
            "roc_auc": "{:.4f}",
            "lift_at_20_vs_base_rate": "{:+.2%}",
            "lift_at_50_vs_base_rate": "{:+.2%}",
            "lift_at_100_vs_base_rate": "{:+.2%}",
        }
    )
)

# ------------------------------------------------------------
# 7. Identify the strongest method
# ------------------------------------------------------------

best_method_at_50 = (
    comparison_table
    .sort_values(
        "precision_at_50",
        ascending=False,
    )
    .iloc[0]["method"]
)

baseline_p50 = float(
    comparison_table.loc[
        comparison_table["method"]
        == "Week 4 rule baseline",
        "precision_at_50",
    ].iloc[0]
)

forest_p50 = float(
    comparison_table.loc[
        comparison_table["method"]
        == "Random Forest",
        "precision_at_50",
    ].iloc[0]
)

print("\nRESULT INTERPRETATION")
print("-" * 70)
print(
    "Best measured method at Precision@50: "
    f"{best_method_at_50}"
)
print(
    "Week 4 baseline Precision@50: "
    f"{baseline_p50:.2%}"
)
print(
    "Random Forest Precision@50: "
    f"{forest_p50:.2%}"
)

if forest_p50 > baseline_p50:
    print(
        "The Random Forest improves on the "
        "baseline at Precision@50 on the "
        "held-out clients."
    )
elif forest_p50 < baseline_p50:
    print(
        "The Week 4 baseline performs better "
        "than the Random Forest at "
        "Precision@50 on the held-out clients."
    )
else:
    print(
        "The Random Forest and baseline match "
        "at Precision@50 on the held-out clients."
    )

print(
    "This is an observed test-set result, "
    "not proof of future production performance."
)

# ------------------------------------------------------------
# 8. Save comparison receipts and ranked queue
# ------------------------------------------------------------

output_dir = (
    repo_root
    / "work"
    / "outputs"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

comparison_path = (
    output_dir
    / "w05_model_comparison.csv"
)

model_queue_path = (
    output_dir
    / "w05_random_forest_ranked_queue.csv"
)

metrics_json_path = (
    output_dir
    / "w05_model_metrics.json"
)

comparison_table.to_csv(
    comparison_path,
    index=False,
)

random_forest_ranked.to_csv(
    model_queue_path,
    index=False,
)

metrics_receipt = {
    "assignment": (
        "ML-08 Capstone Modeling Lane"
    ),
    "lane": (
        "Refresh / Content Opportunity Scoring"
    ),
    "random_seed": RANDOM_SEED,
    "split": (
        "Grouped by client_id, "
        "75 percent train and 25 percent test"
    ),
    "training_rows": int(
        len(X_train)
    ),
    "test_rows": int(
        len(X_test)
    ),
    "training_clients": int(
        len(train_clients)
    ),
    "test_clients": int(
        len(test_clients)
    ),
    "test_base_rate": round(
        test_base_rate,
        6,
    ),
    "comparison": (
        comparison_table
        .round(6)
        .to_dict(
            orient="records"
        )
    ),
}

with open(
    metrics_json_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metrics_receipt,
        file,
        indent=2,
    )

print("\nFILES WRITTEN")
print("-" * 70)
print(f"Comparison table: {comparison_path}")
print(f"Model queue: {model_queue_path}")
print(f"Metrics receipt: {metrics_json_path}")

print("\nTOP 20 RANDOM FOREST RECOMMENDATIONS")
display(
    random_forest_ranked.head(20)
)

WEEK 4 BASELINE ON THE HELD-OUT TEST SET
----------------------------------------------------------------------
Eligible test candidates: 5
Maximum baseline score: 73.94

Training Logistic Regression...
Training Random Forest...
Model training complete.

HONEST COMPARISON TABLE
----------------------------------------------------------------------


,method,test_base_rate,precision_at_20,precision_at_50,precision_at_100,average_precision,roc_auc,lift_at_20_vs_base_rate,lift_at_50_vs_base_rate,lift_at_100_vs_base_rate
0,Week 4 rule baseline,51.65%,45.00%,44.00%,45.00%,0.5170,0.5004,-6.65%,-7.65%,-6.65%
1,Logistic Regression,51.65%,75.00%,76.00%,74.00%,0.5884,0.5858,+23.35%,+24.35%,+22.35%
2,Random Forest,51.65%,50.00%,54.00%,49.00%,0.5848,0.6037,-1.65%,+2.35%,-2.65%



RESULT INTERPRETATION
----------------------------------------------------------------------
Best measured method at Precision@50: Logistic Regression
Week 4 baseline Precision@50: 44.00%
Random Forest Precision@50: 54.00%
The Random Forest improves on the baseline at Precision@50 on the held-out clients.
This is an observed test-set result, not proof of future production performance.

FILES WRITTEN
----------------------------------------------------------------------
Comparison table: /content/flyrank-ml-internship-sunaina/work/outputs/w05_model_comparison.csv
Model queue: /content/flyrank-ml-internship-sunaina/work/outputs/w05_random_forest_ranked_queue.csv
Metrics receipt: /content/flyrank-ml-internship-sunaina/work/outputs/w05_model_metrics.json

TOP 20 RANDOM FOREST RECOMMENDATIONS


,rank,content_id,client_id,actual_label,model_probability
0,1,content_f55fd2d8ed04,client_4e07408562,1,0.8216
1,2,content_0b47dae0c7f9,client_8527a891e2,0,0.8185
2,3,content_e988c1699454,client_8527a891e2,1,0.8176
3,4,content_c148e44db30d,client_8527a891e2,0,0.8173
4,5,content_790cc62d4743,client_8527a891e2,1,0.8143
5,6,content_a5a2fbc76336,client_8527a891e2,0,0.8117
6,7,content_643b51c0a848,client_8527a891e2,1,0.8112
7,8,content_846bb4dd8b44,client_8527a891e2,0,0.8102
8,9,content_569475b335ab,client_4e07408562,1,0.8100
9,10,content_35d63627bf3e,client_8527a891e2,0,0.8097


## 3. Train and compare against my baseline

I compare three ranking methods:

1. **Week 4 rule baseline**
2. **Logistic Regression**
3. **Random Forest**

The Week 4 rule remains unchanged:

- A page is eligible when `days_since_last_update >= 180`.
- It must also have `impressions_90d >= 300`.
- Staleness contributes up to 60 points.
- Search visibility contributes up to 40 points.

For an honest comparison, I calculate the baseline score and model probabilities for the same held-out test rows.

All methods are evaluated with the same target and the same Precision@20, Precision@50, and Precision@100 metrics.

The test-set base rate is also included. This shows how much better or worse each ranked queue is than selecting pages without a useful ranking signal.

The baseline score and predicted probabilities are ranking scores, not guarantees. A high model probability indicates a stronger measured pattern associated with the observed decline label. It does not prove that a refresh will cause improvement.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# ML-08 — SECTION 4
# Error analysis and interpretation
# ============================================================

# ------------------------------------------------------------
# 1. Build prediction-level error table
# ------------------------------------------------------------

forest_predictions = (
    random_forest_test_scores >= 0.50
).astype(int)

error_table = (
    X_test.copy()
    .reset_index(drop=False)
    .rename(
        columns={
            "index": "original_row_index"
        }
    )
)

error_table["actual_label"] = (
    y_test.to_numpy()
)

error_table["predicted_label"] = (
    forest_predictions
)

error_table["predicted_probability"] = (
    random_forest_test_scores
)

error_table["is_correct"] = (
    error_table["actual_label"]
    == error_table["predicted_label"]
)

error_table["error_type"] = np.select(
    [
        (
            error_table["actual_label"] == 0
        )
        & (
            error_table[
                "predicted_label"
            ] == 1
        ),
        (
            error_table["actual_label"] == 1
        )
        & (
            error_table[
                "predicted_label"
            ] == 0
        ),
    ],
    [
        "False positive",
        "False negative",
    ],
    default="Correct",
)

error_table.insert(
    0,
    "test_case_number",
    np.arange(
        1,
        len(error_table) + 1,
    ),
)

# ------------------------------------------------------------
# 2. Overall classification checks
# ------------------------------------------------------------

confusion = confusion_matrix(
    y_test,
    forest_predictions,
)

tn, fp, fn, tp = confusion.ravel()

classification_summary = pd.DataFrame(
    {
        "metric": [
            "Accuracy",
            "Precision at 0.50 threshold",
            "Recall at 0.50 threshold",
            "True negatives",
            "False positives",
            "False negatives",
            "True positives",
        ],
        "value": [
            accuracy_score(
                y_test,
                forest_predictions,
            ),
            precision_score(
                y_test,
                forest_predictions,
                zero_division=0,
            ),
            recall_score(
                y_test,
                forest_predictions,
                zero_division=0,
            ),
            tn,
            fp,
            fn,
            tp,
        ],
    }
)

print("RANDOM FOREST CLASSIFICATION CHECK")
print("-" * 70)
display(classification_summary)

confusion_table = pd.DataFrame(
    confusion,
    index=[
        "Actual not declining",
        "Actual declining",
    ],
    columns=[
        "Predicted not declining",
        "Predicted declining",
    ],
)

print("\nCONFUSION MATRIX")
display(confusion_table)

# ------------------------------------------------------------
# 3. Add value-range buckets
# ------------------------------------------------------------

error_table["staleness_range"] = pd.cut(
    error_table[
        "days_since_last_update"
    ],
    bins=[
        -np.inf,
        30,
        90,
        180,
        365,
        np.inf,
    ],
    labels=[
        "0-30 days",
        "31-90 days",
        "91-180 days",
        "181-365 days",
        "366+ days",
    ],
)

error_table["impression_range"] = pd.cut(
    error_table["impressions_90d"],
    bins=[
        -np.inf,
        99,
        299,
        2999,
        29999,
        np.inf,
    ],
    labels=[
        "0-99",
        "100-299",
        "300-2,999",
        "3,000-29,999",
        "30,000+",
    ],
)

# ------------------------------------------------------------
# 4. Group error summaries
# ------------------------------------------------------------

def make_group_error_table(
    data,
    group_column,
):
    table = (
        data.groupby(
            group_column,
            observed=False,
            dropna=False,
        )
        .agg(
            n=(
                "actual_label",
                "size",
            ),
            actual_decline_rate=(
                "actual_label",
                "mean",
            ),
            predicted_decline_rate=(
                "predicted_label",
                "mean",
            ),
            error_rate=(
                "is_correct",
                lambda values: (
                    1 - values.mean()
                ),
            ),
            false_positives=(
                "error_type",
                lambda values: (
                    values
                    .eq("False positive")
                    .sum()
                ),
            ),
            false_negatives=(
                "error_type",
                lambda values: (
                    values
                    .eq("False negative")
                    .sum()
                ),
            ),
            mean_probability=(
                "predicted_probability",
                "mean",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "error_rate",
                "n",
            ],
            ascending=[
                False,
                False,
            ],
        )
    )

    return table


group_columns = [
    column
    for column in [
        "content_type",
        "main_intent",
        "staleness_range",
        "impression_range",
    ]
    if column in error_table.columns
]

group_error_tables = {}

for column in group_columns:
    group_error_tables[column] = (
        make_group_error_table(
            error_table,
            column,
        )
    )

    print(
        f"\nERRORS BY "
        f"{column.upper()}"
    )
    print("-" * 70)

    display(
        group_error_tables[column]
        .head(15)
        .style.format(
            {
                "actual_decline_rate": "{:.2%}",
                "predicted_decline_rate": "{:.2%}",
                "error_rate": "{:.2%}",
                "mean_probability": "{:.4f}",
            }
        )
    )

# ------------------------------------------------------------
# 5. Permutation feature importance
# ------------------------------------------------------------

print("\nCalculating permutation importance...")

permutation_result = permutation_importance(
    random_forest_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

importance_table = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance_mean": (
            permutation_result
            .importances_mean
        ),
        "importance_std": (
            permutation_result
            .importances_std
        ),
    }
).sort_values(
    "importance_mean",
    ascending=False,
).reset_index(drop=True)

print("\nTOP PERMUTATION FEATURES")
print("-" * 70)
display(
    importance_table.head(10)
)

top_three_features = (
    importance_table
    .head(3)["feature"]
    .tolist()
)

print("\nTOP THREE FEATURES")
for number, feature in enumerate(
    top_three_features,
    start=1,
):
    print(
        f"{number}. {feature}"
    )

# ------------------------------------------------------------
# 6. Explain why the top features are plausible
# ------------------------------------------------------------

feature_explanations = {
    "days_since_last_update": (
        "Staleness may relate to whether information, "
        "examples, or search intent coverage is outdated."
    ),
    "content_age_days": (
        "Older content may have had more time to lose "
        "relevance or face stronger competition."
    ),
    "impressions_90d": (
        "Impressions measure existing search visibility "
        "and the size of the observed opportunity."
    ),
    "clicks_90d": (
        "Clicks measure how much search visibility "
        "currently becomes traffic."
    ),
    "sessions_90d": (
        "Sessions provide measured context about "
        "recent page usage."
    ),
    "ctr": (
        "CTR may reflect how well a search result "
        "matches user interest at its measured position."
    ),
    "avg_position": (
        "Average position relates to search visibility "
        "and the difficulty of earning clicks."
    ),
    "engagement_rate": (
        "Engagement may provide directional evidence "
        "about whether visitors find the page useful."
    ),
    "scroll_rate": (
        "Scroll behaviour may indicate how deeply "
        "visitors consume the page."
    ),
    "content_type": (
        "Different content types can have different "
        "lifecycles and measurement patterns."
    ),
    "main_intent": (
        "Search intent can influence how quickly "
        "content becomes outdated."
    ),
    "word_count": (
        "Content length may reflect page format and "
        "depth, although it does not measure quality."
    ),
    "search_volume": (
        "Search volume provides context about "
        "potential topic demand."
    ),
    "competition": (
        "Competitive topics may change more quickly "
        "and may be harder to maintain."
    ),
}

print("\nPLAUSIBILITY CHECK FOR TOP FEATURES")
print("-" * 70)

for feature in top_three_features:
    explanation = feature_explanations.get(
        feature,
        (
            "This feature may capture measured page "
            "context, but its relationship should be "
            "treated as directional rather than causal."
        ),
    )

    print(
        f"{feature}: {explanation}"
    )

# ------------------------------------------------------------
# 7. Show three concrete wrong cases
# ------------------------------------------------------------

wrong_cases = error_table.loc[
    error_table["error_type"]
    != "Correct"
].copy()

wrong_cases[
    "confidence_of_wrong_prediction"
] = np.where(
    wrong_cases["error_type"]
    .eq("False positive"),
    wrong_cases[
        "predicted_probability"
    ],
    1
    - wrong_cases[
        "predicted_probability"
    ],
)

wrong_cases = wrong_cases.sort_values(
    "confidence_of_wrong_prediction",
    ascending=False,
)


def explain_hard_case(row):
    reasons = []

    if (
        row.get(
            "days_since_last_update",
            0,
        )
        >= 180
    ):
        reasons.append(
            "the page is stale according to the "
            "Week 4 threshold"
        )
    else:
        reasons.append(
            "the page is relatively fresh"
        )

    if (
        row.get(
            "impressions_90d",
            0,
        )
        >= 300
    ):
        reasons.append(
            "it has meaningful measured "
            "search visibility"
        )
    else:
        reasons.append(
            "it has limited measured "
            "search visibility"
        )

    avg_position = row.get(
        "avg_position",
        np.nan,
    )

    if pd.notna(avg_position):
        if avg_position <= 10:
            reasons.append(
                "it has a strong average "
                "search position"
            )
        elif avg_position > 50:
            reasons.append(
                "it ranks deeply in search results"
            )

    ctr_value = row.get(
        "ctr",
        np.nan,
    )

    if pd.notna(ctr_value):
        if ctr_value >= 3:
            reasons.append(
                "its measured CTR is relatively healthy"
            )
        elif ctr_value < 0.5:
            reasons.append(
                "its measured CTR is low"
            )

    return (
        "; ".join(reasons)
        + ". The numeric signals point in different "
        "directions, and the data cannot observe "
        "topic seasonality, page accuracy, business "
        "priority, or recent editorial decisions."
    )


wrong_cases["why_the_case_is_hard"] = (
    wrong_cases.apply(
        explain_hard_case,
        axis=1,
    )
)

wrong_case_columns = [
    column
    for column in [
        "test_case_number",
        "error_type",
        "actual_label",
        "predicted_label",
        "predicted_probability",
        "days_since_last_update",
        "content_age_days",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "sessions_90d",
        "content_type",
        "main_intent",
        "why_the_case_is_hard",
    ]
    if column in wrong_cases.columns
]

three_wrong_cases = (
    wrong_cases
    .head(3)
    .copy()
)

print("\nTHREE CONCRETE WRONG CASES")
print("-" * 70)

if len(three_wrong_cases) == 0:
    print(
        "No threshold-based errors were found."
    )
else:
    display(
        three_wrong_cases[
            wrong_case_columns
        ]
    )

# ------------------------------------------------------------
# 8. Automatic summary of where errors are highest
# ------------------------------------------------------------

print("\nERROR INTERPRETATION SUMMARY")
print("-" * 70)

for column, table in (
    group_error_tables.items()
):
    populated = table.loc[
        table["n"] >= 20
    ].copy()

    if len(populated) == 0:
        continue

    hardest_group = (
        populated.iloc[0]
    )

    print(
        f"Highest measured error group for "
        f"{column}: "
        f"{hardest_group[column]} "
        f"(n={int(hardest_group['n']):,}, "
        f"error rate="
        f"{hardest_group['error_rate']:.2%})."
    )

print(
    "\nThese are observed test-set patterns. "
    "Small groups should not be treated as "
    "stable conclusions."
)

# ------------------------------------------------------------
# 9. Save error analysis receipts
# ------------------------------------------------------------

error_summary_path = (
    output_dir
    / "w05_error_summary.csv"
)

feature_importance_path = (
    output_dir
    / "w05_feature_importance.csv"
)

wrong_cases_path = (
    output_dir
    / "w05_three_wrong_cases.csv"
)

error_table.to_csv(
    error_summary_path,
    index=False,
)

importance_table.to_csv(
    feature_importance_path,
    index=False,
)

three_wrong_cases[
    wrong_case_columns
].to_csv(
    wrong_cases_path,
    index=False,
)

print("\nERROR ANALYSIS FILES WRITTEN")
print("-" * 70)
print(
    f"Prediction and error table: "
    f"{error_summary_path}"
)
print(
    f"Feature importance: "
    f"{feature_importance_path}"
)
print(
    f"Three wrong cases: "
    f"{wrong_cases_path}"
)

# ------------------------------------------------------------
# 10. Final verification
# ------------------------------------------------------------

assert len(
    train_clients.intersection(
        test_clients
    )
) == 0

assert leaked_features == []

assert len(
    comparison_table
) == 3

assert {
    "Week 4 rule baseline",
    "Logistic Regression",
    "Random Forest",
}.issubset(
    set(
        comparison_table["method"]
    )
)

assert comparison_table[
    "precision_at_20"
].between(
    0,
    1,
).all()

assert comparison_table[
    "precision_at_50"
].between(
    0,
    1,
).all()

assert comparison_table[
    "precision_at_100"
].between(
    0,
    1,
).all()

assert comparison_path.exists()
assert model_queue_path.exists()
assert metrics_json_path.exists()
assert error_summary_path.exists()
assert feature_importance_path.exists()
assert wrong_cases_path.exists()

print("\nFINAL SELF-CHECK")
print("-" * 70)
print("Method choice explained: YES")
print("Grouped client split used: YES")
print("Train/test client overlap: NONE")
print("Random seed fixed at 42: YES")
print("Week 4 baseline recomputed on test rows: YES")
print("Logistic Regression trained: YES")
print("Random Forest trained: YES")
print("Same Precision@K metrics used: YES")
print("Test base rate reported: YES")
print("Forbidden label-derived inputs used: NO")
print("Raw IDs used as predictive features: NO")
print("Feature importance inspected: YES")
print("Three concrete wrong cases inspected: YES")
print("Output receipts written: YES")
print("All verification assertions passed.")

print("\nFINAL NOTE")
print("-" * 70)
print(
    "The final ranking is decision support. "
    "It prioritizes pages for human review and "
    "does not prove that a refresh will cause "
    "traffic or engagement to improve."
)

RANDOM FOREST CLASSIFICATION CHECK
----------------------------------------------------------------------


,metric,value
0,Accuracy,0.5806
1,Precision at 0.50 threshold,0.5812
2,Recall at 0.50 threshold,0.6732
3,True negatives,"1,657.0000"
4,False positives,"1,783.0000"
5,False negatives,"1,201.0000"
6,True positives,"2,474.0000"



CONFUSION MATRIX


,Predicted not declining,Predicted declining
Actual not declining,1657,1783
Actual declining,1201,2474



ERRORS BY CONTENT_TYPE
----------------------------------------------------------------------


...


Calculating permutation importance...

TOP PERMUTATION FEATURES
----------------------------------------------------------------------


,feature,importance_mean,importance_std
0,days_with_impressions,0.0218,0.0039
1,avg_position,0.0059,0.0004
2,impressions_90d,0.0047,0.0016
3,position_tier,0.0037,0.0007
4,ctr,0.0028,0.0011
5,content_age_days,0.0024,0.0024
6,clicks_90d,0.0020,0.0004
7,scroll_rate,0.0017,0.0008
8,impression_tier,0.0011,0.0004
9,days_with_sessions,0.0008,0.0005



TOP THREE FEATURES
1. days_with_impressions
2. avg_position
3. impressions_90d

PLAUSIBILITY CHECK FOR TOP FEATURES
----------------------------------------------------------------------
days_with_impressions: This feature may capture measured page context, but its relationship should be treated as directional rather than causal.
avg_position: Average position relates to search visibility and the difficulty of earning clicks.
impressions_90d: Impressions measure existing search visibility and the size of the observed opportunity.

THREE CONCRETE WRONG CASES
----------------------------------------------------------------------


,test_case_number,error_type,actual_label,predicted_label,predicted_probability,days_since_last_update,content_age_days,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,content_type,main_intent,why_the_case_is_hard
6474,6475,False negative,1,0,0.0822,92,238,1,0,0.0000,NaN,1,keyword article,transactional,the page is relatively fresh; it has limited measured search visibility; its measured CTR is low. The numeric signals point in different...
466,467,False negative,1,0,0.1267,20,358,2,0,0.0000,50.0000,2,keyword article,informational,the page is relatively fresh; it has limited measured search visibility; its measured CTR is low. The numeric signals point in different...
2672,2673,False positive,0,1,0.8185,103,238,1191,0,0.0000,23.1000,5,keyword article,informational,the page is relatively fresh; it has meaningful measured search visibility; its measured CTR is low. The numeric signals point in differ...



ERROR INTERPRETATION SUMMARY
----------------------------------------------------------------------
Highest measured error group for content_type: comparison article (n=697, error rate=42.75%).
Highest measured error group for main_intent: commercial (n=1,005, error rate=44.98%).
Highest measured error group for staleness_range: 91-180 days (n=1,218, error rate=45.65%).
Highest measured error group for impression_range: 300-2,999 (n=2,170, error rate=44.56%).

These are observed test-set patterns. Small groups should not be treated as stable conclusions.

ERROR ANALYSIS FILES WRITTEN
----------------------------------------------------------------------
Prediction and error table: /content/flyrank-ml-internship-sunaina/work/outputs/w05_error_summary.csv
Feature importance: /content/flyrank-ml-internship-sunaina/work/outputs/w05_feature_importance.csv
Three wrong cases: /content/flyrank-ml-internship-sunaina/work/outputs/w05_three_wrong_cases.csv

FINAL SELF-CHECK
---------------------

## 4. Errors and interpretation

I inspect the Random Forest because it is the more flexible model and produces the final learned ranking queue.

For classification-style error analysis, I use a probability threshold of 0.50:

- A false positive is a page the model predicts as declining when its observed label is not declining.
- A false negative is a page the model predicts as not declining when its observed label is declining.

The 0.50 threshold is used only to make the errors easier to inspect. The main business output remains a ranking based on predicted probability.

I examine:

- The confusion matrix
- Errors by content type
- Errors by search intent
- Errors by staleness range
- Errors by impression range
- Permutation feature importance
- Three concrete wrong cases

Feature importance describes what the fitted model relies on for this test-set prediction task. It does not establish a causal relationship.

A highly important feature would be suspicious if it directly represented the label or future outcome. The forbidden label-derived columns were removed before training.

Wrong cases may be difficult because historical numeric signals cannot fully represent topic seasonality, content quality, business value, technical SEO issues, editorial plans, or whether a page has recently changed outside the measurement window.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.